In [1]:
from pathlib import Path
while not (Path.cwd() / '.git').exists():
    %cd ..

/home/matthew/study/grab-voc-triage


In [2]:
import pandas as pd
import config
import numpy as np
from sklearn import metrics

In [3]:
df_mini = pd.read_csv(config.MODEL_PREDS_DIR + "/openai/manual_annotation__ver1__gpt-4o-mini.csv")

df_mini.head(2)

,Unnamed: 0,userName,score,at,content,DRIVER_OPS,APP_AND_MAPS,PRICING_AND_BILLING,DRIVER_OPS_pred,APP_AND_MAPS_pred,PRICING_AND_BILLING_pred
0,141,Pengguna Google,1,2026-08-29 11:18:30,gak tau kenapa udah gak kayak dulu lagi pakai ...,NEG,ABSENT,ABSENT,ABSENT,NEG,ABSENT
1,320,Pengguna Google,1,2026-08-22 00:49:40,skrang grab jelek banget driver nya gapernah d...,NEG,ABSENT,ABSENT,NEG,ABSENT,ABSENT


In [4]:
df_4o = pd.read_csv(config.MODEL_PREDS_DIR + "/openai/manual_annotation__ver1__gpt-4o.csv")
df_4o.head(2)

,Unnamed: 0,userName,score,at,content,DRIVER_OPS,APP_AND_MAPS,PRICING_AND_BILLING,DRIVER_OPS_pred,APP_AND_MAPS_pred,PRICING_AND_BILLING_pred
0,141,Pengguna Google,1,2026-08-29 11:18:30,gak tau kenapa udah gak kayak dulu lagi pakai ...,NEG,ABSENT,ABSENT,ABSENT,ABSENT,NEG
1,320,Pengguna Google,1,2026-08-22 00:49:40,skrang grab jelek banget driver nya gapernah d...,NEG,ABSENT,ABSENT,NEG,ABSENT,ABSENT


In [5]:
df_gt = pd.read_csv(config.MANUAL_ANNOTATION_DATA_PATH)
df_gt.head(2)

,Unnamed: 0,userName,score,at,content,DRIVER_OPS,APP_AND_MAPS,PRICING_AND_BILLING
0,141,Pengguna Google,1,2026-08-29 11:18:30,gak tau kenapa udah gak kayak dulu lagi pakai ...,ABSENT,ABSENT,ABSENT
1,320,Pengguna Google,1,2026-08-22 00:49:40,skrang grab jelek banget driver nya gapernah d...,NEG,ABSENT,ABSENT


In [6]:
def classification_report(y_true : np.array, y_pred : np.array) -> str:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    print('--------------------------')
    print('PRIMARY METRIC')
    print('--------------------------')
    for i, cat in enumerate(config.CATEGORIES):
        macro_f1 = metrics.f1_score(y_true[:, i], y_pred[:, i], average = 'macro')
        print(f"{cat} Macro F1-score: {macro_f1}")
    print('--------------------------')
    print('SECONDARY METRIC')
    print('--------------------------')
    print((y_true == y_pred).all(axis = 1).mean())
    print('--------------------------')
    print('DIAGNOSTIC METRIC')
    print('--------------------------')
    for i, cat in enumerate(config.CATEGORIES):
        recalls = metrics.recall_score(y_true[:, i], y_pred[:, i], average = None, labels = ['ABSENT', 'NEG'])

        print(f"{cat} ABSENT Recall: {recalls[0]}")
        print(f"{cat} NEG Recall: {recalls[1]}")
    
classification_report(df_gt[config.CATEGORIES], df_mini[np.array(config.CATEGORIES) + "_pred"])
print()
classification_report(df_gt[config.CATEGORIES], df_4o[np.array(config.CATEGORIES) + "_pred"])

--------------------------
PRIMARY METRIC
--------------------------
DRIVER_OPS Macro F1-score: 0.9255952380952381
APP_AND_MAPS Macro F1-score: 0.9131944444444444
PRICING_AND_BILLING Macro F1-score: 0.9451754385964912
--------------------------
SECONDARY METRIC
--------------------------
0.9
--------------------------
DIAGNOSTIC METRIC
--------------------------
DRIVER_OPS ABSENT Recall: 0.9761904761904762
DRIVER_OPS NEG Recall: 0.875
APP_AND_MAPS ABSENT Recall: 0.9090909090909091
APP_AND_MAPS NEG Recall: 0.9411764705882353
PRICING_AND_BILLING ABSENT Recall: 0.9736842105263158
PRICING_AND_BILLING NEG Recall: 0.9166666666666666

--------------------------
PRIMARY METRIC
--------------------------
DRIVER_OPS Macro F1-score: 0.9255952380952381
APP_AND_MAPS Macro F1-score: 0.9780219780219781
PRICING_AND_BILLING Macro F1-score: 0.9199999999999999
--------------------------
SECONDARY METRIC
--------------------------
0.92
--------------------------
DIAGNOSTIC METRIC
-------------------------

GPT-4o prediction is not so much different in performance to gpt-4o-mini

In [12]:
for c in config.CATEGORIES:
    print(df_gt[c].value_counts())

DRIVER_OPS
ABSENT    42
NEG        8
Name: count, dtype: int64
APP_AND_MAPS
ABSENT    33
NEG       17
Name: count, dtype: int64
PRICING_AND_BILLING
ABSENT    38
NEG       12
Name: count, dtype: int64


based on the data distribution, food category is very small hence will be dropped. Furthermore, the POS also come to a very small percentage of the data. 
Cause generally the positive comment is on overall generic stuff, they mostly clustered to ABSENT. From the business perspective, NEG is much more informative to fix problem in the company. 

In [8]:
false_preds = df_mini[
    ~(df_gt[config.CATEGORIES].to_numpy() == df_mini[np.array(config.CATEGORIES) + "_pred"].to_numpy()).all(axis = 1)
]
for _, p in false_preds.iterrows():
    print(p['Unnamed: 0'])
    print(p.content)
    print(p[config.CATEGORIES + list(np.array(config.CATEGORIES) + "_pred")])
    print()

141
gak tau kenapa udah gak kayak dulu lagi pakai grab hemat tunggu waktu 1 jam tapi sampai 1 5 jam payah banget
DRIVER_OPS                     NEG
APP_AND_MAPS                ABSENT
PRICING_AND_BILLING         ABSENT
DRIVER_OPS_pred             ABSENT
APP_AND_MAPS_pred              NEG
PRICING_AND_BILLING_pred    ABSENT
Name: 0, dtype: object

1496
knpa sya ksi bintang satu karna ada kurir kurang sya suka katanya udh nlpn tpi di cht grab gk ada kan kocak malah saya di marahin baru juga bli dan download apliksi ini lebih parahnya bayarnya di lebih kan di pesanan tidak sgtu lah ini sya buntung 10rb biarkanlah sedekahin aj
DRIVER_OPS                     NEG
APP_AND_MAPS                ABSENT
PRICING_AND_BILLING         ABSENT
DRIVER_OPS_pred                NEG
APP_AND_MAPS_pred              NEG
PRICING_AND_BILLING_pred       NEG
Name: 3, dtype: object

480
lucu deh ni apk ga usah ada option hemat kalo ternyata bisa sembarang dimainin driver ga salah drivernya tp apk nya yg bikin aturan a

In [9]:
df_mini2 = pd.read_csv(config.MODEL_PREDS_DIR + "/openai/manual_annotation__ver2__gpt-4o-mini.csv")

In [10]:
classification_report(df_gt[config.CATEGORIES], df_mini2[np.array(config.CATEGORIES) + "_pred"])

--------------------------
PRIMARY METRIC
--------------------------
DRIVER_OPS Macro F1-score: 0.9607843137254902
APP_AND_MAPS Macro F1-score: 0.9773857982813207
PRICING_AND_BILLING Macro F1-score: 0.9733333333333334
--------------------------
SECONDARY METRIC
--------------------------
0.94
--------------------------
DIAGNOSTIC METRIC
--------------------------
DRIVER_OPS ABSENT Recall: 1.0
DRIVER_OPS NEG Recall: 0.875
APP_AND_MAPS ABSENT Recall: 1.0
APP_AND_MAPS NEG Recall: 0.9411764705882353
PRICING_AND_BILLING ABSENT Recall: 0.9736842105263158
PRICING_AND_BILLING NEG Recall: 1.0


In [11]:
false_preds = df_mini2[
    ~(df_gt[config.CATEGORIES].to_numpy() == df_mini2[np.array(config.CATEGORIES) + "_pred"].to_numpy()).all(axis = 1)
]
for _, p in false_preds.iterrows():
    print(p['Unnamed: 0'])
    print(p.content)
    print(p[config.CATEGORIES + list(np.array(config.CATEGORIES) + "_pred")])
    print()

1496
knpa sya ksi bintang satu karna ada kurir kurang sya suka katanya udh nlpn tpi di cht grab gk ada kan kocak malah saya di marahin baru juga bli dan download apliksi ini lebih parahnya bayarnya di lebih kan di pesanan tidak sgtu lah ini sya buntung 10rb biarkanlah sedekahin aj
DRIVER_OPS                     NEG
APP_AND_MAPS                ABSENT
PRICING_AND_BILLING         ABSENT
DRIVER_OPS_pred                NEG
APP_AND_MAPS_pred           ABSENT
PRICING_AND_BILLING_pred       NEG
Name: 3, dtype: object

936
baru juga daftar kok dh dibilang berkali kali sih
DRIVER_OPS                  ABSENT
APP_AND_MAPS                   NEG
PRICING_AND_BILLING         ABSENT
DRIVER_OPS_pred             ABSENT
APP_AND_MAPS_pred           ABSENT
PRICING_AND_BILLING_pred    ABSENT
Name: 12, dtype: object

684
saya 1jm nunguin pesanan saya di antar malah di batalin gegara gak ada driver yg antar tolong kasih reting buat kita sebagai konsumen biar mereka pada jerah batalin pengantaran jangn cuma bis